# Week 3 Deliverable

In [2]:
import pandas as pd

In [3]:
sold = pd.read_csv("ResidentialSoldFINAL.csv")
list = pd.read_csv("ResidentialListingsFINAL.csv")

/var/folders/y8/n1dk_n9n2fsf3cpk4z474lmc0000gn/T/ipykernel_34402/2235357989.py:1: DtypeWarning: Columns (0,1,9,53,65,66,67) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("ResidentialSoldFINAL.csv")
/var/folders/y8/n1dk_n9n2fsf3cpk4z474lmc0000gn/T/ipykernel_34402/2235357989.py:2: DtypeWarning: Columns (2,39) have mixed types. Specify dtype option on import or set low_memory=False.
  list = pd.read_csv("ResidentialListingsFINAL.csv")


### Fetching Mortgage FRED Dataset

In [4]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']
mortgage

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29
...,...,...
2880,2026-06-11,6.52
2881,2026-06-18,6.47
2882,2026-06-25,6.49
2883,2026-07-02,6.43


### Resample Weekly Rates to Monthly Averages

In [5]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()
mortgage_monthly

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975
...,...,...
659,2026-03,6.1775
660,2026-04,6.3320
661,2026-05,6.4425
662,2026-06,6.4900


### Creating Matching ket on MLS Datasets

In [6]:
# Sold dataset — key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
sold['year_month']

0         2024-01
1         2024-01
2         2024-01
3         2024-01
4         2024-01
           ...   
423851    2026-05
423852    2026-05
423853    2026-05
423854    2026-05
423855    2026-05
Name: year_month, Length: 423856, dtype: period[M]

In [7]:
# Listings dataset — key off ListingContractDate
list['year_month'] = pd.to_datetime(list['ListingContractDate']).dt.to_period('M')
list['year_month']

0         2024-01
1         2024-01
2         2024-01
3         2024-01
4         2024-01
           ...   
591460    2026-05
591461    2026-05
591462    2026-05
591463    2026-05
591464    2026-05
Name: year_month, Length: 591465, dtype: period[M]

### Merge

In [8]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
sold_with_rates

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,year_month,rate_30yr_fixed
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,Other,94401,6472.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,2024-01,6.6425
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,2024-01,6.6425
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,2024-01,6.6425
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,Palm Springs Unified,92262,NaN,13504.0,CRMLS,CRMLS_MLSL,NaN,NaN,2024-01,6.6425
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,Los Angeles Unified,91356,0.0,17873.0,CRMLS,CRMLS_CRM,NaN,NaN,2024-01,6.6425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423851,NaN,VenturaCoastal,Wood,True,NaN,NaN,False,3500000.0,1063517703,NaN,...,NaN,93060,NaN,3117589.2,CRMLS,CRMLS_CRF,NaN,NaN,2026-05,6.4425
423852,Southland,Southland,Wood,True,NaN,NaN,False,675000.0,1052367803,NaN,...,Los Angeles Unified,91345,0.0,7039.0,CRMLS,CRMLS_CRM,NaN,NaN,2026-05,6.4425
423853,ContraCosta,ContraCosta,"Carpet,SeeRemarks,Stone,Wood",NaN,NaN,True,False,35000000.0,1048618285,NaN,...,NaN,94507,3039.0,935669.0,CRMLS,CRMLS_CCBE,NaN,NaN,2026-05,6.4425
423854,BeverlyHillsGreaterLa,SouthBay,NaN,False,NaN,NaN,True,499000.0,1020935560,NaN,...,Los Angeles Unified,91316,504.0,30457.0,CRMLS,CRMLS_CRM,NaN,NaN,2026-05,6.4425


In [9]:
listings_with_rates = list.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,317 E. Bayfront,2024-01,6.6425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591460,1500000.0,1153629738,NaN,NaN,NaN,Thomas,Levin,32.954514,-117.113154,12970 La Tortola,...,False,2.0,NaN,92129,NaN,NaN,7534.0,12970 La Tortola,2026-05,6.4425
591461,13999999.0,1153102507,NaN,2026-06-15,14500000.0,Heather,Rocha,NaN,NaN,5516 LA CRESCENTA,...,False,4.0,NaN,92067,Pacific Sotheby's Int'l Realty,NaN,168577.0,5516 LA CRESCENTA,2026-05,6.4425
591462,637000.0,1151741919,NaN,NaN,NaN,Griselda,Espinoza,33.944936,-118.268552,327 E 101st Street,...,False,NaN,NaN,90003,NaN,NaN,5176.0,327 E 101st Street,2026-05,6.4425
591463,499999.0,1146590110,NaN,NaN,NaN,Kerry,Christenson,33.262347,-116.386675,437 Ocotillo Circle,...,NaN,2.0,Borrego Springs Unified,92004,NaN,0.00,60984.0,437 Ocotillo Circle,2026-05,6.4425


### Validate Merge

In [10]:
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [11]:
print(
    sold_with_rates[
        ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
    ].head()
)


    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-30    2024-01    858000.0           6.6425
4  2024-01-29    2024-01   1890500.0           6.6425
